## Exercise 04. A/B testing
---

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/checking-logs.sqlite')

In [2]:
test_query = """
WITH user_first_views AS (
    SELECT uid, MIN(datetime) as first_view_ts
    FROM pageviews
    GROUP BY uid
)
SELECT 
    CASE 
        WHEN t.first_commit_ts < ufv.first_view_ts THEN 'before'
        ELSE 'after'
    END as time,
    AVG((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0) as avg_diff
FROM test t
JOIN deadlines d ON t.labname = d.labs
JOIN user_first_views ufv ON t.uid = ufv.uid
WHERE t.labname != 'project1'
GROUP BY time
"""

test_results = pd.io.sql.read_sql(test_query, conn)

In [3]:
control_query = """
WITH avg_first_view AS (
    SELECT datetime(AVG(JULIANDAY(first_view_ts)), 'auto') as avg_first_view_ts
    FROM (
        SELECT uid, MIN(datetime) as first_view_ts
        FROM pageviews
        GROUP BY uid
    )
)
SELECT 
    'before' as time,
    AVG((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0) as avg_diff
FROM control t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
  AND t.first_commit_ts < (SELECT avg_first_view_ts FROM avg_first_view)
  
UNION ALL

SELECT 
    'after' as time,
    AVG((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0) as avg_diff
FROM control t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
  AND t.first_commit_ts >= (SELECT avg_first_view_ts FROM avg_first_view)
"""

control_results = pd.io.sql.read_sql(control_query, conn)

In [4]:
print(test_results)
print(control_results)

     time    avg_diff
0   after  103.953446
1  before   61.156632
     time    avg_diff
0  before   99.901448
1   after  113.232346


In [5]:
conn.close()

In [6]:
print("yes, hypothesis is true")

yes, hypothesis is true
